# P2-1 · Bài 1 · Mạng nơ-ron học thế nào

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch
x = torch.randn(128, 3, 32, 32)          # batch ảnh giả
x.shape, x.dtype, x.device               # torch.Size([128, 3, 32, 32]) torch.float32 cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
x = x.to(device)                         # chuyển lên GPU nếu có (model cũng phải ở cùng thiết bị)
x.flatten(1).shape                       # [128, 3072]: duỗi mỗi ảnh thành vector cho MLP

(torch.Size([128, 3, 32, 32]), torch.float32, device(type='cpu'))
torch.Size([128, 3072])


In [ ]:
import torch.nn as nn
model = nn.Sequential(nn.Flatten(), nn.Linear(3072, 256), nn.ReLU(), nn.Linear(256, 10))
sum(p.numel() for p in model.parameters())     # 789258

789258


In [ ]:
x = torch.tensor([1., 2.])
W1 = torch.tensor([[0.5, -0.5], [0.25, 0.75]], requires_grad=True)
w2 = torch.tensor([1., -1.], requires_grad=True)
L = (w2 @ torch.relu(W1 @ x) - 1) ** 2
L.backward()
W1.grad      # tensor([[ 0.0000,  0.0000], [ 5.5000, 11.0000]])
w2.grad      # tensor([-0.0000, -9.6250])

tensor([[ 0.0000,  0.0000],
        [ 5.5000, 11.0000]])
tensor([-0.0000, -9.6250])


In [ ]:
w = torch.tensor(2., requires_grad=True)
for _ in range(2):
    ((3 * w - 1) ** 2).backward()
    print(w.grad)          # 30.0 rồi 60.0, không phải 30.0 hai lần

tensor(30.)
tensor(60.)


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# dữ liệu giả cùng hình dạng CIFAR-10 để chạy thử ngay (CIFAR-10 thật: xem phần thực hành)
# mỗi lớp có một "tông màu" riêng lẫn trong nhiễu mạnh, nên có quy luật để học
palette = torch.randn(10, 3)
def fake_cifar(n):
    y = torch.randint(0, 10, (n,))
    x = torch.randn(n, 3, 32, 32) + 0.08 * palette[y][:, :, None, None]
    return TensorDataset(x, y)
train_loader = DataLoader(fake_cifar(4000), batch_size=128, shuffle=True)
val_loader = DataLoader(fake_cifar(1000), batch_size=256)

model = nn.Sequential(nn.Flatten(), nn.Linear(3072, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()                                  # bật chế độ train (dropout, batchnorm)
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()

    model.eval()                                   # tắt dropout, batchnorm dùng thống kê đã học
    correct = total = 0
    with torch.no_grad():                          # không lưu đồ thị: nhanh hơn, ít bộ nhớ hơn
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.numel()
    print(f"epoch {epoch}: loss {loss.item():.3f}, acc val {correct / total:.3f}")

epoch 0: loss 1.221, acc val 0.577
epoch 1: loss 0.232, acc val 0.599
epoch 2: loss 0.047, acc val 0.600
epoch 3: loss 0.017, acc val 0.596
epoch 4: loss 0.010, acc val 0.595
